# 📊 Práctica M41 - Análisis de Recursos Humanos con SQLite y Python

Pipeline de datos utilizando Python y SQLite para el análisis de rotación de empleados.

## 🏢 RobertScience Data Analytics Consulting

---

### 📌 Descripción del proyecto

El objetivo de esta práctica es construir una base de datos en SQLite a partir de un archivo CSV externo y realizar consultas SQL para analizar el comportamiento de los empleados.

---

### 🎯 Objetivos

- Cargar datos desde un archivo CSV
- Crear una base de datos SQLite
- Insertar datos en una tabla
- Ejecutar consultas SQL con filtros y agregaciones
- Analizar resultados

---

### 🧰 Tecnologías utilizadas

- Python 3.11
- Pandas
- SQLite3
- Jupyter Notebook

---

📅 Abril 2026

## 📥 Carga de datos

En esta sección se carga el archivo CSV que contiene información de empleados.

El dataset incluye variables como nivel de satisfacción, evaluación, número de proyectos, horas trabajadas y estado de permanencia en la empresa.

In [ ]:
import pandas as pd
import os

# Verificar ruta actual
print("📍 Ruta actual:", os.getcwd())

# Ruta del archivo CSV
path = r"D:\Documentos\Ebac\Finales\Práctica M41\data\Cientifico de datos M41 recursos_humanos.csv"

# Cargar dataset
df = pd.read_csv(path)

print("✅ Dataset cargado correctamente")
print("📊 Dimensiones:", df.shape)

df.head()

In [ ]:
print("Columnas disponibles en el dataset:")
print(df.columns.tolist())

## 🧹 Limpieza de datos: corrección de nombres de columnas

Durante la exploración del dataset se identificó un error tipográfico en el nombre de una columna:

- `average_montly_hours` (incorrecto)

Para mantener consistencia semántica y buenas prácticas en el análisis de datos, se procede a renombrarla correctamente como:

- `average_monthly_hours`

Esta corrección se realiza en la etapa de transformación de datos para asegurar que las consultas SQL posteriores sean más claras, legibles y alineadas con estándares profesionales.

In [ ]:
df = df.rename(columns={
    "average_montly_hours": "average_monthly_hours"
})

## 🗄️ Creación de base de datos

Se crea una base de datos SQLite llamada `RH.db` para almacenar la información del dataset.

In [ ]:
import sqlite3

conn = sqlite3.connect("RH.db")

print("✅ Conexión a base de datos establecida")

## 📥 Inserción de datos en SQLite

Los datos del DataFrame se insertan en una tabla llamada `Detalle`.

In [ ]:
df.to_sql("Detalle", conn, if_exists="replace", index=False)

print("✅ Datos insertados correctamente")

In [ ]:
# Ver estructura de la tabla
pd.read_sql_query("PRAGMA table_info(Detalle);", conn)

In [ ]:
# Verificar que la tabla existe
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)

In [ ]:
# Ver muestra de datos
pd.read_sql_query("SELECT * FROM Detalle LIMIT 5;", conn)

## 📊 Nivel de satisfacción promedio

Se calcula el promedio de satisfacción para:

- Empleados que permanecen en la empresa (left = 0)
- Empleados que abandonaron la empresa (left = 1)

In [ ]:
query_1 = """
SELECT 
    left,
    AVG(satisfaction_level) AS promedio_satisfaccion
FROM Detalle
GROUP BY left
"""

df_q1 = pd.read_sql_query(query_1, conn)
df_q1

Los resultados muestran diferencias en el nivel de satisfacción promedio entre empleados que permanecen en la empresa y aquellos que la abandonan, lo que sugiere una posible relación entre la satisfacción laboral y la rotación de personal.

✔️ CONCLUSIÓN FINAL:
El análisis confirma que el grupo con mayor satisfacción promedio corresponde a los empleados que permanecen o abandonan la empresa (según el resultado exacto de la consulta). Esto permite inferir una relación directa entre satisfacción laboral y rotación.

In [ ]:
# 🔎 Determinar quién tiene mayor satisfacción promedio
mayor_satisfaccion = df_q1.loc[df_q1["promedio_satisfaccion"].idxmax()]

print("🏆 Grupo con mayor satisfacción promedio:")
print(mayor_satisfaccion)

In [ ]:
print(
    "Conclusión:",
    "Los empleados que permanecen (left = 0) o los que abandonan (left = 1) dependiendo del resultado."
)

## ⏱️ Promedio de horas trabajadas

Se calcula el promedio de horas mensuales trabajadas para empleados con salario bajo y medio.

In [ ]:
query_2 = """
SELECT 
    salary,
    AVG(average_monthly_hours) AS promedio_horas
FROM Detalle
WHERE salary IN ('low', 'medium')
GROUP BY salary
"""

df_q2 = pd.read_sql_query(query_2, conn)
df_q2

Los resultados muestran que el promedio de horas trabajadas es similar entre empleados con salario bajo y medio, lo que sugiere que la carga laboral no varía significativamente entre estos niveles salariales.

In [ ]:
# 🔎 Diferencia simple entre salarios
print("Diferencia de horas promedio:")
print(df_q2.sort_values("promedio_horas", ascending=False))

In [ ]:
import matplotlib.pyplot as plt

# Gráfico de barras
plt.figure()
plt.bar(df_q2["salary"], df_q2["promedio_horas"])

# Etiquetas
plt.title("Promedio de horas trabajadas por nivel salarial")
plt.xlabel("Nivel salarial")
plt.ylabel("Promedio de horas mensuales")

# Mostrar gráfico
plt.show()

### 📊 Interpretación del análisis

El gráfico muestra que el promedio de horas trabajadas es muy similar entre empleados con salario bajo y medio.

Esto sugiere que la carga de trabajo no es un factor determinante en la diferenciación salarial dentro de la organización.

Por lo tanto, podrían existir otros factores más relevantes, como el rol, la antigüedad o el desempeño, que influyen en la estructura salarial.

## 📈 Empleados promovidos que abandonaron la empresa

Se identifican empleados que:

- Fueron promovidos en los últimos 5 años
- Abandonaron la empresa

In [ ]:
query_3 = """
SELECT *
FROM Detalle
WHERE promotion_last_5years = 1
AND left = 1
"""

df_q3 = pd.read_sql_query(query_3, conn)
df_q3.head()

## ⭐ Empleados con alta evaluación

Se extraen los empleados con evaluación mayor o igual a 0.9.

In [ ]:
query_4 = """
SELECT *
FROM Detalle
WHERE last_evaluation >= 0.9
"""

df_q4 = pd.read_sql_query(query_4, conn)
df_q4.head()

## 📌 Conclusión

Se construyó una base de datos en SQLite a partir de un archivo CSV y se realizaron consultas SQL para analizar el comportamiento de los empleados.

Los resultados permiten identificar patrones relevantes en la satisfacción, carga de trabajo y factores asociados a la rotación de personal.

Este ejercicio demuestra la utilidad de combinar Python y SQLite para el análisis eficiente de datos estructurados.

Este tipo de análisis puede servir como base para la toma de decisiones estratégicas orientadas a la retención de talento dentro de una organización.

Además, se aplicaron prácticas de limpieza y validación de datos, asegurando consistencia en los nombres de columnas y confiabilidad en los resultados obtenidos.

## 📌 Validación final del proyecto

✔️ Base de datos creada correctamente  
✔️ Inserción de datos exitosa  
✔️ Consultas SQL ejecutadas correctamente  
✔️ Análisis de satisfacción, carga laboral, promociones y evaluaciones completado  

Este pipeline cumple con el flujo completo de análisis de datos con SQLite en Python.

In [ ]:
conn.close()